In [1]:
import pandas as pd
import requests

In [2]:
def fetch_ibge_data():
    """Consome a API do IBGE e retorna um DataFrame padronizado com id_municipio e uf."""
    url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    response = requests.get(url)
    dados = response.json()
    
    dict_ufs = {'11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
                '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL',
                '28':'SE','29':'BA','31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR',
                '42':'SC','43':'RS','50':'MS','51':'MT','52':'GO','53':'DF'}
    
    municipios = [{'id_municipio': str(m['id']), 'nome_municipio': m['nome'], 
                   'uf': dict_ufs.get(str(m['id'])[:2], 'NA')} for m in dados]
    return pd.DataFrame(municipios)

def build_abt(df_gold, df_censo, df_atlas):
    """Cruza as bases e gera a Tabela Analítica (ABT) pronta para modelagem."""
    df_gold['id_municipio'] = df_gold['id_municipio'].astype(str)
    df_censo['id_municipio'] = df_censo['id_municipio'].astype(str)
    df_atlas['id_municipio'] = df_atlas['id_municipio'].astype(str)
    
    df_master = pd.merge(df_gold, df_censo, on=['id_municipio', 'rede'], how='left')
    df_master = pd.merge(df_master, df_atlas, on='id_municipio', how='left')
    
    return df_master